In [ ]:
import pandas as pd 
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt 
from PIL import Image
from pathlib import Path

In [ ]:
lab_flat_fields = {'T': 'lab_flat_field_target.fits', 
                   'G_OLD': 'lab_flat_field_global_old.fits', 
                   'G': 'lab_flat_field_global.fits'}  
cal_reference = 'obs_cal_info.csv'
cal_channel_and_sample_indices = {
    # col 0 also affected but they don't interpolate it bc it's in the dark area 
    'detector_array_tap_columns': {
        'T': [160, 320, 480],
        'G': [80, 160, 240]
    },
    'filter_seam_channels': {
        'T': [40, 41, 115],
        'G': [12, 49]
    },
    'masked_columns_split': {
        'TL': [0, 1, 2, 3, 4, 5, 6, 7],
        'TR': [636, 637, 638, 639],
        'GL': [0, 1, 2, 3], # could do one less? they don't say. 
        'GR': [318, 319]
    },
    'masked_columns': {
        'T': [0, 1, 2, 3, 4, 5, 6, 7, 636, 637, 638, 639],
        'G': [0, 1, 2, 3, 318, 319]
    },
    'scattered_light_columns_split': {
        'TL': [8, 9, 10, 11, 12, 13, 14],
        'TR': [627, 628, 629, 630, 631, 632, 633, 634, 635],
        'GL': [4, 5, 6, 7],
        'GR': [314, 315, 316, 317]
    },
    
    'scattered_light_columns': {
        'T': [8, 9, 10, 11, 12, 13, 14, 627, 628, 629, 630, 631,
              632, 633, 634, 635],
        'G': [4, 5, 6, 7, 314, 315, 316, 317],
    },
    'channels_omitted_at_level1b': {
        'T': [0],
        'G': [0, 1, 2, 3]
    },
}



In [ ]:
def make_dark_signal_mean_image(dark_l0_path: str, dark_cols: list = None): 
    """
    The dark signal of an observation is estimated from a dark signal observation
    acquired prior to the real observation during a non-illuminated portion of
    the orbit. They were also used to generate the bad detector element image 
    (BDE), but we are not currently doing that. 

    The dark signal is averaged for all "lines" to get an average dark signal
    value for each cross-track sample and spectral channel. 

    dark_cols: for when we don't want to subtract avg dark signal from the cols 
    used for dark pedestal corrections later on (pixels with purportedly no light landing 
    on them during an obs) 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape

    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the mean 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_avg = dark_obs_data[:, 2:-2, :].mean(axis=1)

    if dark_cols is not None: 
        # set avg for dark cols to 0 to preserve dark signal values for later steps
        dark_signal_avg[:, dark_cols] = 0
        
    # main obs data is in int16  
    return dark_signal_avg.astype(np.int16) 


def make_dark_signal_std_image(dark_l0_path: str, dark_cols: list = None):
    """
    Not a pipeline step at the moment. Just for QA. Could be used for new BDE. 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape

    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the std 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_std = np.std(dark_obs_data.transpose(1, 0, 2)[2:-2, :, :], axis=0)

    return dark_signal_std


def detector_array_tap_interpolation(obs_data: np.ndarray, cols: list):
    """
    Columns related to the read-out are linearly interpolated in the spatial 
    direction (columns on either side). The columns are 161, 321, 481 for
    target and 81, 161, 241 for global mode.
    """
    cols = np.array(cols)
    obs_data[:, :, cols] = (obs_data[:, :, cols - 1] + obs_data[:, :, cols + 1]) / 2.0
    return obs_data
    

def filter_seam_interpolation(obs_data: np.ndarray, channels: list):
    """
    Order sorting filter seams show up on the detector, we linearly interpolate
    across them in the spectral direction (these are horizontal features). For 
    target mode they are channels 41, 42, and 116 and for global mode they are 
    13 and 50.
    """
    for channel in channels:
        # we have special cases for 41 & 42 because they are next to each other 
        # this could be less specific if we anticipate expanding the number of channels
        # flagged for this, but I don't think we will. We also could get rid of the 
        # check if the pipeline is never used for target mode obs. 
        if channel == 41:
            obs_data[:, 41, :] = (2/3) * obs_data[:, 40, :] + (1/3) * obs_data[:, 43, :]
        elif channel == 42:
            obs_data[:, 42, :] = (1/3) * obs_data[:, 40, :] + (2/3) * obs_data[:, 43, :]
        else:
            obs_data[:, channel, :] = (obs_data[:, channel - 1, :] + obs_data[:, channel + 1, :]) / 2.0
    return obs_data
    

def bad_detector_element_correction(dss_image: np.ndarray, bde_path: str): 
    """
    Elements are flagged 0-5 in these fits files. Values 1-4 seem to indicate 
    things that are "flagged", 1 being the lowest level and 4 the highest. I think
    the mission interpolated everything flagged 1-4 in the spectral direction. 
    This does seem a little problematic to me because the filter seams (channel features) 
    are flagged, usually as 4, so right now we interpolate across them twice. 

    Pixels with only one "good" pixel above or below it (this means they are likely 
    near the edge) will adopt that good pixel's value. If there are NO good pixels in 
    a column, right now we don't do anything. This could change, but should only 
    matter for the tap columns we interpolate across in the spatial direction in a later 
    step. 
    
    """
    import numpy as np
    from astropy.io import fits

    with fits.open(bde_path) as hdul:
        bde_map = hdul[0].data
    
    bad_mask = bde_map != 0  
    n_rows, n_cols = bad_mask.shape  # 86 x 320 for global mode 

    # we only need to figure out the interpolation weughts once per detector "frame" 
    # where: 
    # new_val = val_at_top + (bad_row-top_row)/(bottom_row-top_row)*(val_at_bottom-val_at_top)
    # where the "weight" is (bad_row-top_row)/(bottom_row-top_row)
    
    bad_rows, bad_cols = np.where(bad_mask)  # all bad pixels 
    
    # find closest top and bottom pixel
    top_rows = np.full(len(bad_rows), -1, dtype=np.intp)
    bottom_rows = np.full(len(bad_rows), n_rows, dtype=np.intp)

    # "good" pixels per column 
    good_in_col = []
    for c in range(n_cols):
        good_in_col.append(np.where(~bad_mask[:, c])[0])

    # find closest good pixels 
    for i, (r, c) in enumerate(zip(bad_rows, bad_cols)):
        g = good_in_col[c]
        if g.size == 0:
            continue  # idk what to do if the whole column is bad. for readout cols 
                      # we interpolate across the col (spatially) instead of across 
                      # channels (spectrally) 
        idx = np.searchsorted(g, r)         
        if idx > 0:
            top_rows[i]    = g[idx - 1] # above
        if idx < len(g):
            bottom_rows[i] = g[idx] # below
            
    has_top = top_rows >= 0
    has_bottom = bottom_rows <  n_rows
    both = has_top & has_bottom
    top_only = has_top & ~has_bottom
    bottom_only = ~has_top & has_bottom
    # neither = ~has_top & ~has_bottom  # we ignore this right now, 
                                        # could interpolate across / spatially

    # weights for linear interpolation when there is a top and bottom pixel
    denominator = np.where(both, bottom_rows - top_rows, 1).astype(np.float32)
    t = np.where(both, (bad_rows - top_rows) / denominator, 0.0).astype(np.float32)

    if both.any():
        br = bad_rows[both]
        bc = bad_cols[both]
        tr = top_rows[both]
        btr = bottom_rows[both]
        w = t[both]         

        top_vals = dss_image[:, tr, bc].astype(np.float32)   
        bot_vals = dss_image[:, btr, bc].astype(np.float32)  

        interp = top_vals + w[np.newaxis, :] * (bot_vals - top_vals)
        # apply to all frames, should we round differently or leave as a float? idk. 
        dss_image[:, br, bc] = np.round(interp).astype(dss_image.dtype)

    # use only valid neighbor's value for pixels who only have a top or bottom neighbor 
    if top_only.any():
        br = bad_rows[top_only]
        bc = bad_cols[top_only]
        tr = top_rows[top_only]
        # apply to all frames
        dss_image[:, br, bc] = dss_image[:, tr, bc]

    if bottom_only.any():
        br = bad_rows[bottom_only]
        bc = bad_cols[bottom_only]
        btr = bottom_rows[bottom_only]
        # apply to all frames
        dss_image[:, br, bc] = dss_image[:, btr, bc]

    return dss_image 

    
def get_cal_paths(obs: str, local_dir: str): 
    """
    Check spreadsheet containing cal filenames pulled from L1B header files 
    in the PDS3 data. We want to use the latest version available, bc it's
    possible they changed cal files for later versions of the data. 

    Let's also check their validity (ie is this a bad bde / bad dark, how 
    many pixels are flagged, do the cal files exist). 
    """
    cal_files = pd.read_csv(f"{local_dir}/obs_to_cal_file_mapping/{cal_reference}")
    
    cal_files = cal_files[cal_files['obs_id'] == obs.upper()] 

    if len(cal_files) == 0: 
        print("This is not a valid observation with cal file mappings.")  

    # we want the highest version of the observation (ie V03 not V02) 
    cal_files = cal_files.loc[cal_files['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()]

    cal_paths = {}

    data_dir = local_dir+"/data/"
    
    temp = cal_files['obs_temperature']

    mode = "global" if cal_files['obs_type'] == "G" else "target"
    
    beta_angle = cal_files['obs_beta_angle'] 
    
    cal_paths['dark'] = f"{data_dir}/{cal_files['dark_signal_id'].lower()}_l0.fits"
    cal_paths['obs_flat'] = f"{data_dir}/{cal_files['flat_field_id'].lower()}_ff.fits"
    cal_paths['lab_flat'] = f"{local_dir}/cal_files/{lab_flat_fields[cal_files['obs_type']]}"
    cal_paths['bde'] = f"{data_dir}{cal_files['bad_detector_map_id'].lower()}_bde.fits"
    cal_paths['obs'] = f"{data_dir}{cal_files['obs_id'].lower()}_l0.fits"

    # warnings about the cal files, if applicable 
    print(f"Obs {obs} is a {mode} observation conducted at {temp} K and\
     beta angle {beta_angle} during observational period {cal_files['obs_period']}.")

    if cal_files['bad_bde'] or cal_files['bad_dark'] == True: 
        # only applies to a handful of observations, eventually we should 
        # designate alteratives 
        print(f"A dark signal observation used in calibrating this observation was\
              mispointed and should not be used.")

    if not (cal_files['flat_in_usgs'] and 
            cal_files['bde_in_usgs'] and 
            cal_files['dark_in_usgs'] and 
            cal_files['cal_l0_in_usgs']):
        # this only applies to a hanfdul of observations 
        print(f"One of the cal files required is missing from the USGS PDS.") 

    # we may went to return temp and beta angle for cal in the future? 
    for key, filename in cal_paths.items():
        if Path(filename).exists():
            continue 
        else: 
            print(f"{filename} not found in {data_dir}.")
        
    return cal_paths, cal_files['obs_type']

def l0_to_l1b(obs: str, local_dir: str, write_intermediates: bool = False): 
    """
    String together steps 1-9 eventually. 
    """
    from pathlib import Path

    # make folder for saving output & intermediate steps 
    Path(f"{local_dir}/outputs").mkdir(exist_ok=True)
    
    obs = obs.lower() 
    
    # check file mappings for cal files based on obs number, print obs temp 
    # also check if it's a valid obs we have file mappings for 
    cal_paths, mode = get_cal_paths(obs, local_dir)

    # load obs data and convert to "detector pov" orientation to make cal file 
    # application easier 
    print("loading obs data") 
    with fits.open(cal_paths['obs']) as hdul:
            obs_data = hdul[0].data  
    obs_data = obs_data.transpose(1, 0, 2) 

    ### STEP 1: DARK SIGNAL SUBTRACTION 
    # get average dark signal
    print("subtracting dark signal")
    dark_signal_avg = make_dark_signal_mean_image(
        cal_paths['dark'], 
        cal_channel_and_sample_indices['masked_columns'][mode]
    )

    if write_intermediates: 
        fits.writeto(
            f"{local_dir}/outputs/{obs}_dark_avg.fits", 
            dark_signal_avg, 
            overwrite=True
        )
        
    # make dark signal subtracted image (DSS) 
    obs_data -= dark_signal_avg

    if write_intermediates: 
        fits.writeto(
            f"{local_dir}/outputs/{obs}_after_dss.fits",
            obs_data, 
            overwrite=True
        )

    del dark_signal_avg 

    ### STEP 2: BAD DETECTOR ELEMENT CORRECTION
    # interpolate bad detector elements 
    print("interpolating across bad detector elements") 
    obs_data = bad_detector_element_correction(obs_data, cal_paths['bde'])

    if write_intermediates: 
        fits.writeto(
            f"{local_dir}/outputs/{obs}_after_bde.fits",
            obs_data, 
            overwrite=True
        )

    ### STEP 3: DETECTOR ARRAY TAP INTERPOLATION
    print("running detector array tap interpolation") 

    obs_data = detector_array_tap_interpolation(
        obs_data,
        cal_channel_and_sample_indices['detector_array_tap_columns'][mode]
    ) 

    if write_intermediates: 
        fits.writeto(
            f"{local_dir}/outputs/{obs}_after_tap_interp.fits",
            obs_data, 
            overwrite=True
        )

    ### STEP 4: FILTER SEAM INTERPOLATION
    print("running filter seam interpolation") 

    obs_data = filter_seam_interpolation(
        obs_data, 
        cal_channel_and_sample_indices['filter_seam_channels'][mode]
    ) 

    if write_intermediates: 
        fits.writeto(
            f"{local_dir}/outputs/{obs}_after_filter_seam.fits",
            obs_data, 
            overwrite=True
        )

    ## STEP 5: Electronic Panel Ghost Correction 
    # not done yet 

    ## STEP 6: Dark Pedestal Shift Correction
    # not done yet 

    ## STEP 7: Scattered light correction 
    # not done yet 

    ## STEP 8: Laboratory Flat Field 
    # not done yet, super easy 
    # load lab flat and multiply 

    ## STEP 9: Imagine Based Flat Field 
    # not done yet 

    ## STEP 10: Radiometric Calibration 
    # not done yet 

    ## STEP 11: Smooth Shape Correction 
    # not done yet 

    return obs_data 

In [ ]:
obs = "m3g20090125t172601"
local_dir = "/home/bekah/m3-pipeline-dev"
data =  l0_to_l1b(obs, local_dir)

In [ ]:
fits.writeto(
            f"{local_dir}/outputs/{obs}_end_pipeline.fits",
            data, 
            overwrite=True
        )